# Lesson 4: Classification and Neural Networks

In Lesson 3, we predicted numbers (house prices). The model output could be any value: $250,000, $347,821, etc.

**Classification is different.** Instead of "how much?", we're asking "which one?" The answer is a category, not a number.

Before diving into the math, let's see classification in action across three very different domains.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
DATA_PATH = Path("../../../../data")

## Demo 1: Medical Diagnosis

A doctor looks at a breast tissue sample and needs to decide: **benign or malignant?**

This is life-or-death classification. The model takes measurements of cell nuclei (radius, texture, smoothness, etc.) and predicts whether a tumor is cancerous.

Let's see how well a simple classifier can do this:

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# Load the breast cancer dataset
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

print("Breast Cancer Wisconsin Dataset")
print("="*50)
print(f"Samples: {len(X)}")
print(f"Features: {len(cancer.feature_names)} measurements per sample")
print(f"Classes: {cancer.target_names}")
print(f"\nSample features: {cancer.feature_names[:5]}...")

In [ ]:
# Train a classifier
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

# Evaluate
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"\nAccuracy: {accuracy:.1%}")
print(f"\nThis means: out of {len(X_test)} tissue samples, the model correctly")
print(f"identified {int(accuracy * len(X_test))} as benign or malignant.")

# Confusion matrix
cm = confusion_matrix(y_test, predictions)
print(f"\nConfusion Matrix:")
print(f"                 Predicted")
print(f"                 Malignant  Benign")
print(f"Actual Malignant    {cm[0,0]:3d}      {cm[0,1]:3d}")
print(f"       Benign       {cm[1,0]:3d}      {cm[1,1]:3d}")

## Demo 2: Fraud Detection

A credit card company processes millions of transactions daily. Each one needs an instant decision: **legitimate or fraudulent?**

The challenge: fraud is rare (~0.2% of transactions), so the model must be very precise to avoid blocking legitimate purchases while still catching fraud.

In [ ]:
# Load credit card fraud dataset
# Download: kaggle datasets download -d mlg-ulb/creditcardfraud
fraud_df = pd.read_csv(DATA_PATH / "creditcard" / "creditcard.csv")

print("Credit Card Fraud Dataset")
print("="*50)
print(f"Transactions: {len(fraud_df):,}")
print(f"Fraudulent: {fraud_df['Class'].sum():,} ({fraud_df['Class'].mean():.2%})")
print(f"Legitimate: {(fraud_df['Class']==0).sum():,}")
print(f"\nFeatures: {fraud_df.shape[1]-1} (anonymized for privacy)")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score

# Use a sample for speed
fraud_sample = fraud_df.sample(n=20000, random_state=42)
X_fraud = fraud_sample.drop('Class', axis=1)
y_fraud = fraud_sample['Class']

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fraud, y_fraud, test_size=0.2, random_state=42
)

# Train a fraud detector
fraud_model = RandomForestClassifier(n_estimators=50, random_state=42)
fraud_model.fit(X_train_f, y_train_f)

# Evaluate
fraud_preds = fraud_model.predict(X_test_f)
accuracy = accuracy_score(y_test_f, fraud_preds)
precision = precision_score(y_test_f, fraud_preds)
recall = recall_score(y_test_f, fraud_preds)

print(f"\nResults on test set:")
print(f"Accuracy: {accuracy:.1%}")
print(f"Precision: {precision:.1%} (of transactions flagged as fraud, how many actually were)")
print(f"Recall: {recall:.1%} (of actual fraud, how many did we catch)")

## Demo 3: Handwritten Digit Recognition

You write a check, and the bank needs to read your handwriting. Each digit image → one of 10 categories (0-9).

This is **multi-class classification**: more than two possible outputs.

In [ ]:
from sklearn.datasets import load_digits

# Load the digits dataset
digits = load_digits()
X_digits, y_digits = digits.data, digits.target

print("Handwritten Digits Dataset")
print("="*50)
print(f"Images: {len(X_digits)}")
print(f"Image size: 8x8 pixels = 64 features")
print(f"Classes: 0, 1, 2, 3, 4, 5, 6, 7, 8, 9")

# Show some examples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f'Label: {digits.target[i]}')
    ax.axis('off')
plt.suptitle('Sample Handwritten Digits', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Train a digit classifier
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_digits, y_digits, test_size=0.2, random_state=42
)

digit_model = LogisticRegression(max_iter=5000)
digit_model.fit(X_train_d, y_train_d)

accuracy = digit_model.score(X_test_d, y_test_d)
print(f"Accuracy: {accuracy:.1%}")
print(f"\nThe model correctly identifies {accuracy:.1%} of handwritten digits!")

In [ ]:
# Show some predictions
sample_indices = [10, 25, 42, 88, 100]
fig, axes = plt.subplots(1, 5, figsize=(12, 3))

for ax, idx in zip(axes, sample_indices):
    img = digits.images[idx]
    true_label = digits.target[idx]
    pred_label = digit_model.predict([digits.data[idx]])[0]
    
    ax.imshow(img, cmap='gray')
    color = 'green' if pred_label == true_label else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label}', color=color)
    ax.axis('off')

plt.suptitle('Model Predictions (green=correct, red=wrong)', fontsize=12)
plt.tight_layout()
plt.show()

## The Common Thread

Three completely different problems:

| Domain | Input | Output | Stakes |
|--------|-------|--------|--------|
| Medical | Cell measurements | Benign / Malignant | Life or death |
| Finance | Transaction data | Legitimate / Fraud | Money |
| OCR | Pixel values | 0-9 | Convenience |

But the same fundamental approach:
1. Collect labeled examples (inputs with correct outputs)
2. Train a model to find patterns
3. Use those patterns to classify new data

In regression, we predicted numbers. Now we're predicting categories. How do we adapt the same gradient descent approach to this new problem?

That's what the rest of this lesson is about. We'll build up from logistic regression (the simplest classifier) to neural networks (which can learn more complex patterns).

## Our Working Example: The Titanic

For the rest of this lesson, we'll use the Titanic dataset — a classic binary classification problem.

**Why Titanic?**
- Binary classification (survived / died) is simpler to understand
- The features are intuitive (gender, age, ticket class)
- The patterns are real and meaningful ("women and children first")
- It's small enough to visualize everything

We'll build three models of increasing sophistication:
1. **Logistic Regression**: The baseline classifier
2. **Neural Network (MLP)**: More powerful, can learn curves
3. Understanding why neural networks can do things logistic regression can't

In [ ]:
# === VISUALIZATION HELPERS (click to expand) ===
# These functions create the visualizations used in this notebook.
# The code is hidden to keep focus on the ML concepts.

def plot_classification_problem(ages, fares, survived):
    """Show the fundamental classification problem: two groups that need separating."""
    fig, ax = plt.subplots(figsize=(10, 8))
    
    colors = ['#e74c3c' if s == 0 else '#3498db' for s in survived]
    ax.scatter(ages, fares, c=colors, alpha=0.6, edgecolors='white', s=60)
    
    # Legend
    ax.scatter([], [], c='#3498db', s=60, label='Survived', edgecolors='white')
    ax.scatter([], [], c='#e74c3c', s=60, label='Died', edgecolors='white')
    ax.legend(loc='upper right', fontsize=12)
    
    ax.set_xlabel('Age', fontsize=12)
    ax.set_ylabel('Fare (ticket price)', fontsize=12)
    ax.set_title('The Classification Problem: Can You Separate These Groups?', fontsize=14)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_manual_boundary(ages, fares, survived, slope=None, intercept=None):
    """Show what a decision boundary does — separate the space into regions."""
    fig, ax = plt.subplots(figsize=(10, 8))
    
    colors = ['#e74c3c' if s == 0 else '#3498db' for s in survived]
    ax.scatter(ages, fares, c=colors, alpha=0.6, edgecolors='white', s=60)
    
    if slope is not None and intercept is not None:
        # Draw the boundary line
        x_line = np.array([0, 80])
        y_line = slope * x_line + intercept
        ax.plot(x_line, y_line, 'k--', linewidth=2, label='Decision Boundary')
        
        # Shade regions
        ax.fill_between(x_line, y_line, 600, alpha=0.1, color='blue', label='Predict: Survived')
        ax.fill_between(x_line, 0, y_line, alpha=0.1, color='red', label='Predict: Died')
    
    ax.set_xlim(0, 80)
    ax.set_ylim(0, 300)
    ax.set_xlabel('Age', fontsize=12)
    ax.set_ylabel('Fare (ticket price)', fontsize=12)
    ax.set_title('A Decision Boundary Divides the Space', fontsize=14)
    ax.legend(loc='upper right', fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_boundary_accuracy(ages, fares, survived, slope, intercept):
    """Show how a boundary classifies points and compute accuracy."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Predict based on boundary
    predicted = (fares > slope * ages + intercept).astype(int)
    correct = (predicted == survived)
    accuracy = correct.mean()
    
    # Left: show correct/incorrect
    for ax_idx, (ax, title) in enumerate([(axes[0], 'Correct Predictions'), 
                                           (axes[1], 'Incorrect Predictions')]):
        mask = correct if ax_idx == 0 else ~correct
        
        colors = ['#e74c3c' if s == 0 else '#3498db' for s in survived[mask]]
        ax.scatter(ages[mask], fares[mask], c=colors, alpha=0.7, edgecolors='white', s=60)
        
        x_line = np.array([0, 80])
        y_line = slope * x_line + intercept
        ax.plot(x_line, y_line, 'k--', linewidth=2)
        ax.fill_between(x_line, y_line, 600, alpha=0.1, color='blue')
        ax.fill_between(x_line, 0, y_line, alpha=0.1, color='red')
        
        ax.set_xlim(0, 80)
        ax.set_ylim(0, 300)
        ax.set_xlabel('Age', fontsize=12)
        ax.set_ylabel('Fare', fontsize=12)
        count = mask.sum()
        ax.set_title(f'{title}: {count} passengers ({count/len(survived):.0%})', fontsize=13)
        ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'This boundary achieves {accuracy:.1%} accuracy', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    
    return accuracy

def plot_survival_rates(data):
    """Show survival rates by different factors."""
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    # By sex
    survival_by_sex = data.groupby('Sex')['Survived'].mean()
    colors = ['#e74c3c' if x == 'female' else '#3498db' for x in survival_by_sex.index]
    axes[0].bar(survival_by_sex.index, survival_by_sex.values, color=colors)
    axes[0].set_ylabel('Survival Rate')
    axes[0].set_title('By Gender')
    axes[0].set_ylim(0, 1)
    for i, v in enumerate(survival_by_sex.values):
        axes[0].text(i, v + 0.02, f'{v:.0%}', ha='center', fontweight='bold')
    
    # By class
    survival_by_class = data.groupby('Pclass')['Survived'].mean()
    axes[1].bar(['1st', '2nd', '3rd'], survival_by_class.values, color=['#2ecc71', '#f39c12', '#e74c3c'])
    axes[1].set_ylabel('Survival Rate')
    axes[1].set_title('By Ticket Class')
    axes[1].set_ylim(0, 1)
    for i, v in enumerate(survival_by_class.values):
        axes[1].text(i, v + 0.02, f'{v:.0%}', ha='center', fontweight='bold')
    
    # By age group
    data_copy = data.copy()
    data_copy['AgeGroup'] = pd.cut(data_copy['Age'], bins=[0, 12, 18, 35, 50, 100], labels=['Child', 'Teen', 'Young Adult', 'Adult', 'Senior'])
    survival_by_age = data_copy.groupby('AgeGroup', observed=True)['Survived'].mean()
    axes[2].bar(range(len(survival_by_age)), survival_by_age.values, color='#9b59b6')
    axes[2].set_xticks(range(len(survival_by_age)))
    axes[2].set_xticklabels(survival_by_age.index, rotation=45)
    axes[2].set_ylabel('Survival Rate')
    axes[2].set_title('By Age Group')
    axes[2].set_ylim(0, 1)
    
    # Combined: Sex + Class
    survival_combined = data.groupby(['Sex', 'Pclass'])['Survived'].mean().unstack()
    x = np.arange(2)
    width = 0.25
    axes[3].bar(x - width, survival_combined[1], width, label='1st Class', color='#2ecc71')
    axes[3].bar(x, survival_combined[2], width, label='2nd Class', color='#f39c12')
    axes[3].bar(x + width, survival_combined[3], width, label='3rd Class', color='#e74c3c')
    axes[3].set_xticks(x)
    axes[3].set_xticklabels(['Female', 'Male'])
    axes[3].set_ylabel('Survival Rate')
    axes[3].set_title('Gender + Class Combined')
    axes[3].set_ylim(0, 1)
    axes[3].legend()
    
    plt.tight_layout()
    plt.show()

def plot_linear_regression_problem():
    """Show why linear regression fails for classification."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: Linear regression on binary data
    jitter_x = X_simple + np.random.normal(0, 0.05, len(X_simple))
    jitter_y = y_simple + np.random.normal(0, 0.05, len(y_simple))
    
    axes[0].scatter(jitter_x, jitter_y, alpha=0.3, c=y_simple, cmap='RdYlBu')
    
    x_line = np.linspace(-0.5, 1.5, 100)
    y_line = w * x_line + b
    axes[0].plot(x_line, y_line, 'r-', linewidth=2, label=f'y = {w:.2f}x + {b:.2f}')
    
    axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[0].axhline(y=1, color='gray', linestyle='--', alpha=0.5)
    axes[0].set_xlim(-0.5, 1.5)
    axes[0].set_ylim(-0.3, 1.3)
    axes[0].set_xlabel('Is Male (0=Female, 1=Male)')
    axes[0].set_ylabel('Survived')
    axes[0].set_title('Linear Regression on Classification Data')
    axes[0].legend()
    axes[0].set_xticks([0, 1])
    axes[0].set_xticklabels(['Female', 'Male'])
    
    # Right: What happens with extreme values
    ages = titanic['Age'].dropna().values
    survived = titanic.loc[titanic['Age'].notna(), 'Survived'].values
    
    age_mean = ages.mean()
    surv_mean = survived.mean()
    w_age = np.sum((ages - age_mean) * (survived - surv_mean)) / np.sum((ages - age_mean)**2)
    b_age = surv_mean - w_age * age_mean
    
    axes[1].scatter(ages, survived + np.random.normal(0, 0.03, len(survived)), alpha=0.3, c=survived, cmap='RdYlBu')
    
    x_ext = np.linspace(-20, 100, 100)
    y_ext = w_age * x_ext + b_age
    axes[1].plot(x_ext, y_ext, 'r-', linewidth=2, label='Linear regression')
    
    axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[1].axhline(y=1, color='gray', linestyle='--', alpha=0.5)
    axes[1].fill_between(x_ext, 1, 1.5, alpha=0.2, color='red', label='Invalid: > 1')
    axes[1].fill_between(x_ext, -0.5, 0, alpha=0.2, color='red', label='Invalid: < 0')
    
    axes[1].set_xlim(-20, 100)
    axes[1].set_ylim(-0.3, 1.3)
    axes[1].set_xlabel('Age')
    axes[1].set_ylabel('Predicted "Survival"')
    axes[1].set_title('Problem: Predictions Go Outside [0, 1]')
    axes[1].legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

def plot_sigmoid():
    """Show the sigmoid function and how it creates a boundary."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    z = np.linspace(-10, 10, 200)
    s = sigmoid(z)
    
    # Left: The sigmoid curve
    axes[0].plot(z, s, 'b-', linewidth=2)
    axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    axes[0].axhline(y=0, color='gray', alpha=0.3)
    axes[0].axhline(y=1, color='gray', alpha=0.3)
    axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    
    axes[0].fill_between(z[z < 0], 0, s[z < 0], alpha=0.2, color='red')
    axes[0].fill_between(z[z > 0], s[z > 0], 1, alpha=0.2, color='blue')
    
    axes[0].annotate('z < 0 → probability < 0.5\n→ predict Class 0', xy=(-6, 0.15), fontsize=10,
                     bbox=dict(boxstyle='round', facecolor='#ffcccc', alpha=0.8))
    axes[0].annotate('z > 0 → probability > 0.5\n→ predict Class 1', xy=(2, 0.85), fontsize=10,
                     bbox=dict(boxstyle='round', facecolor='#cce5ff', alpha=0.8))
    axes[0].annotate('z = 0 is the\ndecision boundary!', xy=(0.3, 0.55), fontsize=10,
                     bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    
    axes[0].set_xlabel('z (the linear combination)', fontsize=12)
    axes[0].set_ylabel('σ(z) = probability', fontsize=12)
    axes[0].set_title('Sigmoid: Where z=0 Is the Decision Boundary', fontsize=13)
    axes[0].set_ylim(-0.1, 1.1)
    axes[0].grid(True, alpha=0.3)
    
    # Right: Compare linear vs sigmoid
    z2 = np.linspace(-3, 3, 100)
    axes[1].plot(z2, z2, 'r--', linewidth=2, label='Linear: y = z', alpha=0.7)
    axes[1].plot(z2, sigmoid(z2), 'b-', linewidth=2, label='Sigmoid: y = σ(z)')
    axes[1].axhline(y=0, color='gray', alpha=0.3)
    axes[1].axhline(y=1, color='gray', alpha=0.3)
    axes[1].fill_between(z2, 0, 1, alpha=0.1, color='green', label='Valid probability range')
    
    axes[1].set_xlabel('z', fontsize=12)
    axes[1].set_ylabel('Output', fontsize=12)
    axes[1].set_title('Linear Goes Anywhere; Sigmoid Stays in [0, 1]', fontsize=13)
    axes[1].legend(loc='upper left')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_training_curves(train_losses, val_losses):
    """Plot training and validation loss curves."""
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Training Loss', color='#3498db')
    plt.plot(val_losses, label='Validation Loss', color='#e74c3c')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (Cross-Entropy)')
    plt.title('Logistic Regression Training')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_decision_boundary_logistic(X, y, w, b, title):
    """Plot decision boundary for logistic regression."""
    fig, ax = plt.subplots(figsize=(10, 8))
    
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    
    grid = np.c_[xx.ravel(), yy.ravel()]
    probs = logistic_regression_predict(grid, w, b).reshape(xx.shape)
    
    contour = ax.contourf(xx, yy, probs, levels=50, cmap='RdYlBu', alpha=0.6)
    ax.contour(xx, yy, probs, levels=[0.5], colors='black', linewidths=2, linestyles='--')
    
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlBu', edgecolors='black', s=50, alpha=0.8)
    
    plt.colorbar(contour, ax=ax, label='P(Survived)')
    ax.set_xlabel('Age (normalized)')
    ax.set_ylabel('Fare (normalized)')
    ax.set_title(title)
    
    ax.annotate('Decision boundary\n(where P=0.5)', xy=(0, -0.5), fontsize=10,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

def plot_comparison(X, y, w_log, b_log, mlp_model, title1, title2, acc1, acc2):
    """Compare logistic regression and MLP decision boundaries."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    for ax, model_type, title, acc in [(axes[0], 'logistic', title1, acc1), 
                                        (axes[1], 'mlp', title2, acc2)]:
        x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
        y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
        grid = np.c_[xx.ravel(), yy.ravel()]
        
        if model_type == 'logistic':
            probs = logistic_regression_predict(grid, w_log, b_log).reshape(xx.shape)
        else:
            mlp_model.eval()
            with torch.no_grad():
                probs = mlp_model(torch.FloatTensor(grid)).numpy().reshape(xx.shape)
        
        contour = ax.contourf(xx, yy, probs, levels=50, cmap='RdYlBu', alpha=0.6)
        ax.contour(xx, yy, probs, levels=[0.5], colors='black', linewidths=2, linestyles='--')
        ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlBu', edgecolors='black', s=50, alpha=0.8)
        
        plt.colorbar(contour, ax=ax, label='P(Survived)')
        ax.set_xlabel('Age (normalized)')
        ax.set_ylabel('Fare (normalized)')
        ax.set_title(f'{title}\n(Accuracy: {acc:.1%})')
    
    plt.tight_layout()
    plt.show()

def plot_relu():
    """Show the ReLU function and why non-linearity matters."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    x = np.linspace(-5, 5, 100)
    
    axes[0].plot(x, np.maximum(0, x), 'b-', linewidth=2)
    axes[0].axhline(y=0, color='gray', alpha=0.3)
    axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    axes[0].set_xlabel('Input')
    axes[0].set_ylabel('Output')
    axes[0].set_title('ReLU: max(0, x)')
    axes[0].grid(True, alpha=0.3)
    axes[0].annotate('Negative → 0', xy=(-3, 0.2), fontsize=11,
                     bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    axes[0].annotate('Positive → unchanged', xy=(1.5, 3.5), fontsize=11,
                     bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    axes[1].text(0.5, 0.8, 'Without ReLU:', fontsize=14, fontweight='bold',
                 transform=axes[1].transAxes, ha='center')
    axes[1].text(0.5, 0.65, 'Linear → Linear = Linear', fontsize=12,
                 transform=axes[1].transAxes, ha='center', family='monospace')
    axes[1].text(0.5, 0.55, '(Still just a straight line!)', fontsize=11,
                 transform=axes[1].transAxes, ha='center', color='red')
    
    axes[1].text(0.5, 0.35, 'With ReLU:', fontsize=14, fontweight='bold',
                 transform=axes[1].transAxes, ha='center')
    axes[1].text(0.5, 0.2, 'Linear → ReLU → Linear = Non-linear', fontsize=12,
                 transform=axes[1].transAxes, ha='center', family='monospace')
    axes[1].text(0.5, 0.1, '(Can learn curves!)', fontsize=11,
                 transform=axes[1].transAxes, ha='center', color='green')
    
    axes[1].axis('off')
    axes[1].set_title('Why Non-linearity Matters')
    
    plt.tight_layout()
    plt.show()

## The Titanic Dataset

The Titanic disaster of 1912 is one of history's most infamous shipwrecks. Of the 2,224 passengers and crew aboard, more than 1,500 died. But survival wasn't random — it was heavily influenced by factors like gender, ticket class, and age.

Our goal: given information about a passenger, predict whether they survived.

This is a **binary classification** problem. Two groups: survivors and non-survivors. Our job is to find a boundary that separates them.

In [ ]:
titanic = pd.read_csv(DATA_PATH / "titanic" / "titanic.csv")
print(f"Dataset: {len(titanic)} passengers")
titanic.head(10)

## Visualizing the Problem

Before we build any model, let's see what classification actually looks like. We'll plot two features — Age and Fare — and color each passenger by whether they survived.

Look at this plot and ask yourself: **How would you draw a line to separate the blue dots (survived) from the red dots (died)?**

In [ ]:
# Get passengers with known age and reasonable fare (for cleaner visualization)
titanic_viz = titanic.dropna(subset=['Age'])
titanic_viz = titanic_viz[titanic_viz['Fare'] < 300]  # Remove a few extreme outliers

ages = titanic_viz['Age'].values
fares = titanic_viz['Fare'].values
survived = titanic_viz['Survived'].values

print(f"Plotting {len(ages)} passengers")
plot_classification_problem(ages, fares, survived)

It's messy, right? The blue and red dots are mixed together — there's no perfect line that separates them completely. That's the reality of most classification problems.

But we can still find a **decision boundary** — a line that does a *reasonable* job of separating the groups. Everything on one side we predict "survived", everything on the other side we predict "died".

Let me draw an arbitrary boundary and show you what it does:

In [ ]:
# Let's draw a simple boundary: Fare = 50 (horizontal line)
# Anyone above pays more → predict survived, below → predict died
plot_manual_boundary(ages, fares, survived, slope=0, intercept=50)

This boundary says: "If your fare was above $50, predict survived. Otherwise, predict died."

It's a simple rule, and it captures something real — passengers who paid more for tickets were more likely to survive (they were in better cabins, closer to lifeboats).

But how good is this boundary? Let's measure it:

In [ ]:
accuracy = plot_boundary_accuracy(ages, fares, survived, slope=0, intercept=50)

## From Manual Rules to Learning

Our simple "fare > $50" boundary gets about 62% accuracy. Not great — barely better than guessing "died" for everyone (which would get ~62% since 62% of passengers died).

We could try other boundaries:
- Fare > $30?
- Age < 10? (children first)
- Some diagonal line combining age and fare?

But manually trying boundaries is tedious. What we really want is an algorithm that **learns the best boundary from the data**.

This is exactly what classification models do. They look at the training data and figure out where to draw the line.

But before we build a classifier, let's understand what patterns exist in the data that could help us draw a good boundary.

## What Predicts Survival?

Let's look at survival rates broken down by different factors. These patterns tell us which features are most useful for separating the groups.

In [ ]:
print(f"Overall survival rate: {titanic['Survived'].mean():.1%}\n")
plot_survival_rates(titanic)

Strong patterns emerge:

- **Gender is the strongest signal**: 74% of women survived vs only 19% of men ("women and children first")
- **Class matters**: 63% of 1st class survived vs 24% of 3rd class
- **The combination is even more powerful**: A woman in 1st class had 97% survival; a man in 3rd class had only 14%

These patterns tell us where the decision boundary should go. A good classifier should learn: "If female AND 1st class → very likely survived. If male AND 3rd class → very likely died."

Now let's build a model that learns this boundary automatically.

## Building a Classifier: First Attempt with Linear Regression

In the previous lesson, we built linear regression models. They predict: $\hat{y} = wx + b$

Could we just use that here? Let's try predicting survival using just one feature — whether the passenger was male.

In [ ]:
# Prepare simple data: Sex (0=female, 1=male) -> Survived (0 or 1)
titanic_simple = titanic[['Sex', 'Survived']].copy()
titanic_simple['Sex_numeric'] = (titanic_simple['Sex'] == 'male').astype(int)

X_simple = titanic_simple['Sex_numeric'].values
y_simple = titanic_simple['Survived'].values

# Fit linear regression: y = w*x + b
X_mean = X_simple.mean()
y_mean = y_simple.mean()

w = np.sum((X_simple - X_mean) * (y_simple - y_mean)) / np.sum((X_simple - X_mean)**2)
b = y_mean - w * X_mean

print(f"Linear regression: y = {w:.3f}*x + {b:.3f}")
print(f"\nPredictions:")
print(f"  Female (x=0): {b:.3f}")
print(f"  Male (x=1):   {w + b:.3f}")

The predictions are 0.74 for females and 0.19 for males. These happen to be the actual survival rates! But there's a fundamental problem with using linear regression for classification...

In [ ]:
plot_linear_regression_problem()

**The problem:** Linear regression can output any number from -∞ to +∞. But for classification, we need outputs between 0 and 1 that we can interpret as probabilities.

Look at the right plot — the regression line goes outside the valid range. What does "-30% chance of survival" even mean?

We need a way to **squash** any number into the range [0, 1]. Enter the sigmoid function.

## The Sigmoid Function

The **sigmoid function** takes any number and squashes it to a value between 0 and 1:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

- Large negative → close to 0
- Large positive → close to 1
- Zero → exactly 0.5

This gives us a probability we can use for classification.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Test it with some values
test_values = [-10, -5, -2, -1, 0, 1, 2, 5, 10]
print("Input z  →  sigmoid(z)  →  Prediction")
print("-" * 45)
for z in test_values:
    prob = sigmoid(z)
    pred = "Class 1 (survived)" if prob >= 0.5 else "Class 0 (died)"
    print(f"{z:6}   →  {prob:.4f}       →  {pred}")

In [ ]:
plot_sigmoid()

## Logistic Regression = Linear + Sigmoid

**In one sentence:** Logistic regression computes a weighted sum of your features, passes it through sigmoid to get a probability, then classifies using a threshold (usually 0.5).

The formula:
1. Compute: $z = w_1 x_1 + w_2 x_2 + ... + w_n x_n + b$
2. Squash to probability: $\hat{y} = \sigma(z)$
3. Classify: if $\hat{y} \geq 0.5$, predict class 1

That's it. The model learns the weights w and bias b during training.

## Preparing the Data

Let's prepare the Titanic data for training. We'll use four key features.

In [ ]:
# Select features and handle missing values
features = ['Pclass', 'Sex', 'Age', 'Fare']

# Create a clean dataset
data = titanic[features + ['Survived']].copy()
data['Sex'] = (data['Sex'] == 'male').astype(int)  # Convert to 0/1
data = data.dropna()  # Drop rows with missing Age

print(f"Clean dataset: {len(data)} passengers")
print(f"Features: {features}")
data.head()

In [ ]:
# Split into features (X) and target (y)
X = data[features].values
y = data['Survived'].values

# Split into training and validation sets
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalize features (important for gradient descent)
X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0)
X_train_norm = (X_train - X_mean) / X_std
X_val_norm = (X_val - X_mean) / X_std

print(f"Training set: {len(X_train)} samples")
print(f"Validation set: {len(X_val)} samples")

## Training with Gradient Descent

We train using gradient descent — same as regression. The model adjusts weights to minimize a loss function.

For classification, we use **cross-entropy loss** instead of MSE. The intuition: it heavily penalizes confident wrong predictions. If you predict 0.99 (very confident "survived") but they actually died, that's a huge penalty.

In [ ]:
def logistic_regression_predict(X, w, b):
    """Compute predictions: sigmoid(X @ w + b)"""
    z = X @ w + b
    return sigmoid(z)

def compute_loss(y_true, y_pred):
    """Binary cross-entropy loss"""
    epsilon = 1e-15  # Avoid log(0)
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

def compute_gradients(X, y_true, y_pred):
    """Compute gradients for w and b"""
    m = len(y_true)
    error = y_pred - y_true
    dw = (1/m) * X.T @ error
    db = (1/m) * np.sum(error)
    return dw, db

In [ ]:
# Initialize weights
n_features = X_train_norm.shape[1]
w = np.zeros(n_features)
b = 0.0

# Training hyperparameters
learning_rate = 0.1
n_epochs = 200

# Track losses
train_losses = []
val_losses = []

# Training loop
for epoch in range(n_epochs):
    # Forward pass
    y_pred_train = logistic_regression_predict(X_train_norm, w, b)
    y_pred_val = logistic_regression_predict(X_val_norm, w, b)
    
    # Compute loss
    train_loss = compute_loss(y_train, y_pred_train)
    val_loss = compute_loss(y_val, y_pred_val)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    # Compute gradients
    dw, db = compute_gradients(X_train_norm, y_train, y_pred_train)
    
    # Update weights (gradient descent!)
    w = w - learning_rate * dw
    b = b - learning_rate * db
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

print("\nTraining complete!")

In [ ]:
plot_training_curves(train_losses, val_losses)

## Evaluating the Model

Let's see how well our learned decision boundary separates the classes.

In [ ]:
# Make predictions
y_pred_train = logistic_regression_predict(X_train_norm, w, b)
y_pred_val = logistic_regression_predict(X_val_norm, w, b)

# Convert probabilities to class predictions (threshold = 0.5)
y_pred_train_class = (y_pred_train >= 0.5).astype(int)
y_pred_val_class = (y_pred_val >= 0.5).astype(int)

# Calculate accuracy
train_accuracy = np.mean(y_pred_train_class == y_train)
val_accuracy = np.mean(y_pred_val_class == y_val)

print(f"Training Accuracy: {train_accuracy:.1%}")
print(f"Validation Accuracy: {val_accuracy:.1%}")
print(f"\nMuch better than our manual 'fare > $50' boundary ({accuracy:.1%})!")

## What Did the Model Learn?

Remember, the decision boundary is where $z = w_1 x_1 + w_2 x_2 + w_3 x_3 + w_4 x_4 + b = 0$.

The learned weights tell us how each feature influences which side of the boundary a passenger falls on:

In [ ]:
print("Learned weights (what moves you toward 'survived' side of boundary):")
print("-" * 60)
for feature, weight in zip(features, w):
    if weight > 0:
        effect = "↑ increases survival probability"
    else:
        effect = "↓ decreases survival probability"
    print(f"  {feature:10s}: {weight:+.3f}  {effect}")
print(f"  {'Bias':10s}: {b:+.3f}")

The model learned sensible patterns:
- **Sex** (male=1) has the largest negative weight → being male strongly pushes you toward the "died" side
- **Pclass** has negative weight → higher class number (3rd class) pushes toward "died"
- **Age** has small negative weight → older passengers slightly more likely to die
- **Fare** has positive weight → higher fare pushes toward "survived"

This matches the historical patterns we saw earlier! The model learned where to draw the boundary.

## Visualizing the Learned Boundary

Let's see the decision boundary in 2D. We'll train a simpler model using just Age and Fare so we can visualize it.

In [ ]:
# Train a 2D model for visualization
X_2d = data[['Age', 'Fare']].values
y_2d = data['Survived'].values

X_train_2d, X_val_2d, y_train_2d, y_val_2d = train_test_split(X_2d, y_2d, test_size=0.2, random_state=42)

# Normalize
X_mean_2d = X_train_2d.mean(axis=0)
X_std_2d = X_train_2d.std(axis=0)
X_train_2d_norm = (X_train_2d - X_mean_2d) / X_std_2d
X_val_2d_norm = (X_val_2d - X_mean_2d) / X_std_2d

# Train
w_2d = np.zeros(2)
b_2d = 0.0

for epoch in range(500):
    y_pred = logistic_regression_predict(X_train_2d_norm, w_2d, b_2d)
    dw, db = compute_gradients(X_train_2d_norm, y_train_2d, y_pred)
    w_2d = w_2d - 0.1 * dw
    b_2d = b_2d - 0.1 * db

val_acc_2d = np.mean((logistic_regression_predict(X_val_2d_norm, w_2d, b_2d) >= 0.5) == y_val_2d)
print(f"2D model validation accuracy: {val_acc_2d:.1%}")

In [ ]:
plot_decision_boundary_logistic(
    X_train_2d_norm, y_train_2d, w_2d, b_2d,
    f'Logistic Regression: The Learned Decision Boundary'
)

## The Limitation: Straight Boundaries (Unless You Engineer Features)

Look at that decision boundary — it's a **straight line**.

With the features we gave it (Age, Fare), logistic regression draws a straight boundary. You *could* add polynomial features (Age², Fare², Age×Fare) to get curves — but then you're manually figuring out what combinations matter.

What if we could have a model that **learns useful feature combinations automatically**? 

That's exactly what neural networks do, and the "automatic" part is the part a lot of people have a hard time accepting. It's the magic, if you will.

## Enter PyTorch: The Deep Learning Library

To build neural networks, we'll use **PyTorch** — the industry-standard library for deep learning. Before we dive into the code, let's understand what makes it different from NumPy.

**Why not just use NumPy?**

Remember in logistic regression, we computed gradients manually? For a simple model with 5 weights, that was manageable. But neural networks have hundreds or thousands of weights, and the gradients involve chain rules through multiple layers.

PyTorch solves this with **automatic differentiation**: it tracks all operations on your data and automatically computes gradients when you ask. You write the forward pass, PyTorch figures out the backward pass.

**The key concepts:**

| Concept | What it is |
|---------|------------|
| `Tensor` | Like a NumPy array, but can track gradients and run on GPU |
| `nn.Module` | Base class for all neural network components |
| `nn.Linear` | A layer that does `output = input @ weights + bias` |
| `nn.Sequential` | Stack multiple layers in order |
| `loss.backward()` | Compute all gradients automatically |
| `optimizer.step()` | Update all weights using those gradients |

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

print(f"PyTorch version: {torch.__version__}")

# Tensors are like NumPy arrays
numpy_array = np.array([1.0, 2.0, 3.0])
tensor = torch.tensor([1.0, 2.0, 3.0])

print(f"\nNumPy array: {numpy_array}")
print(f"PyTorch tensor: {tensor}")
print(f"Same operations work: tensor * 2 = {tensor * 2}")

The magic happens when we convert our training data to tensors — PyTorch will track all operations and compute gradients for us.

For a quick comparison, here's how you'd train a neural network with **scikit-learn** (familiar, but limited):

In [ ]:
from sklearn.neural_network import MLPClassifier

# sklearn makes it look easy...
sklearn_mlp = MLPClassifier(hidden_layer_sizes=(16,), max_iter=500, random_state=42)
sklearn_mlp.fit(X_train_norm, y_train)

sklearn_acc = sklearn_mlp.score(X_val_norm, y_val)
print(f"sklearn MLP accuracy: {sklearn_acc:.1%}")
print(f"\nThat was easy! But sklearn hides everything.")
print(f"We can't see the weights updating, can't customize the training loop,")
print(f"can't add GPU acceleration, can't build complex architectures.")
print(f"\nFor real deep learning, we need PyTorch.")

Scikit-learn's `MLPClassifier` works, but it's a black box. With PyTorch, we'll see exactly what's happening — and that understanding is what lets you build more sophisticated models later.

Let's build the same network in PyTorch:

## The Multi-Layer Perceptron (MLP)

**In one sentence:** An MLP adds hidden layers that learn to transform your raw features into more useful combinations, then does classification on those learned features.

```
Input (4 features)
       ↓
Hidden Layer (learns feature combinations)
       ↓
Output (probability)
```

The key difference from logistic regression: instead of us manually creating features like Age² or Age×Fare, the hidden layer figures out what combinations are useful.

In [ ]:
class MLP(nn.Module):
    """A simple neural network with one hidden layer."""
    
    def __init__(self, n_features, n_hidden=16):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_features, n_hidden),  # Input → Hidden
            nn.ReLU(),                         # Activation function - this is super important, but we'll talk more about it later.
            nn.Linear(n_hidden, 1),            # Hidden → Output
            nn.Sigmoid()                       # Squash to probability - look, we're using sigmoid again! The reason is because we're doing binary classification. There are other activation functions that can do stuff with our output. 
        )
    
    def forward(self, x):
        return self.layers(x)

# Create the model
mlp = MLP(n_features=4, n_hidden=16)
print(mlp)

In [ ]:
# How many parameters does each model have?
logistic_params = len(features) + 1  # weights + bias
mlp_params = sum(p.numel() for p in mlp.parameters())

print(f"Logistic regression: {logistic_params} parameters")
print(f"MLP (16 hidden):     {mlp_params} parameters")
print(f"\nThe MLP has {mlp_params // logistic_params}x more parameters — more capacity to learn complex patterns.")

In [ ]:
# Convert our numpy data to PyTorch tensors
X_train_t = torch.FloatTensor(X_train_norm)
y_train_t = torch.FloatTensor(y_train).reshape(-1, 1)
X_val_t = torch.FloatTensor(X_val_norm)
y_val_t = torch.FloatTensor(y_val).reshape(-1, 1)

print(f"Training data: {X_train_t.shape}")
print(f"Validation data: {X_val_t.shape}")

## Training the MLP

The training loop looks almost identical to logistic regression:
1. Forward pass: compute predictions
2. Calculate loss
3. Backward pass: compute gradients
4. Update weights

PyTorch handles the gradient computation automatically — we just call `loss.backward()`.

In [ ]:
# Fresh model
mlp = MLP(n_features=4, n_hidden=16)

# Loss function and optimizer
loss_fn = nn.BCELoss()  # Binary cross-entropy (same as we used before)
optimizer = optim.Adam(mlp.parameters(), lr=0.01)

# Track losses
mlp_train_losses = []
mlp_val_losses = []

# Training loop
n_epochs = 200

for epoch in range(n_epochs):
    # Forward pass
    mlp.train()
    y_pred = mlp(X_train_t)
    loss = loss_fn(y_pred, y_train_t)
    
    # Backward pass
    optimizer.zero_grad()  # Clear old gradients
    loss.backward()        # Compute new gradients
    optimizer.step()       # Update weights
    
    # Track losses
    mlp_train_losses.append(loss.item())
    
    mlp.eval()
    with torch.no_grad():
        val_loss = loss_fn(mlp(X_val_t), y_val_t)
        mlp_val_losses.append(val_loss.item())
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d}: Train Loss = {loss.item():.4f}, Val Loss = {val_loss.item():.4f}")

print("\nTraining complete!")

In [ ]:
# Evaluate the MLP
mlp.eval()
with torch.no_grad():
    mlp_train_pred = (mlp(X_train_t) >= 0.5).float()
    mlp_val_pred = (mlp(X_val_t) >= 0.5).float()
    
    mlp_train_acc = (mlp_train_pred == y_train_t).float().mean().item()
    mlp_val_acc = (mlp_val_pred == y_val_t).float().mean().item()

print("="*55)
print("MODEL COMPARISON")
print("="*55)
print(f"{'Model':<25} {'Train Acc':<15} {'Val Acc':<15}")
print("-"*55)
print(f"{'Logistic Regression':<25} {train_accuracy:<15.1%} {val_accuracy:<15.1%}")
print(f"{'MLP (16 hidden neurons)':<25} {mlp_train_acc:<15.1%} {mlp_val_acc:<15.1%}")
print("="*55)

## The MLP's Decision Boundary

Let's see what the MLP learns compared to logistic regression. We'll train on the same 2D data (Age and Fare) and visualize both decision boundaries side by side.

In [ ]:
# Train MLP on 2D data for visualization
X_train_2d_t = torch.FloatTensor(X_train_2d_norm)
y_train_2d_t = torch.FloatTensor(y_train_2d).reshape(-1, 1)
X_val_2d_t = torch.FloatTensor(X_val_2d_norm)
y_val_2d_t = torch.FloatTensor(y_val_2d).reshape(-1, 1)

mlp_2d = MLP(n_features=2, n_hidden=16)
optimizer_2d = optim.Adam(mlp_2d.parameters(), lr=0.01)

for epoch in range(500):
    mlp_2d.train()
    pred = mlp_2d(X_train_2d_t)
    loss = loss_fn(pred, y_train_2d_t)
    optimizer_2d.zero_grad()
    loss.backward()
    optimizer_2d.step()

mlp_2d.eval()
with torch.no_grad():
    mlp_2d_val_acc = ((mlp_2d(X_val_2d_t) >= 0.5).float() == y_val_2d_t).float().mean().item()

print(f"MLP (2D) validation accuracy: {mlp_2d_val_acc:.1%}")

In [ ]:
plot_comparison(
    X_train_2d_norm, y_train_2d, w_2d, b_2d, mlp_2d,
    'Logistic Regression', 'Neural Network (MLP)',
    val_acc_2d, mlp_2d_val_acc
)

print("Left: Logistic regression with raw features — straight boundary.")
print("Right: Neural network — learns its own features, can capture more complex patterns.")

## Why Does the MLP Learn Curved Boundaries?

The hidden layer **creates new features** from the inputs — combinations that make the problem easier to solve.

Instead of us manually adding Age², Fare², Age×Fare, the network figures out what combinations are useful. Each neuron in the hidden layer detects a different pattern. The output layer then combines these learned features.

Think of it this way: the hidden layer transforms the data into a new space where the classes become easier to separate.

### The Role of ReLU

Between layers, we apply **ReLU** (Rectified Linear Unit): keep positive values, zero out negatives.

Why does this matter? Without it, stacking linear layers just gives another linear layer — you'd still get straight boundaries. ReLU introduces the **non-linearity** that lets networks learn curves.

We'll explore this more in the next lesson when we trace actual numbers through the network.

In [ ]:
plot_relu()

### Quick Terminology

Before moving on, let's name the parts:

- **Layer**: A transformation step (input → hidden → output)
- **Neuron**: One unit in a layer — our hidden layer has 16 neurons
- **Weights/Parameters**: The numbers the network learns (we have 97 of them)
- **Activation function**: The non-linearity (ReLU, Sigmoid) that enables curve-learning

That's enough vocabulary to understand what's happening. The next lesson goes deeper into the mechanics.

## What's Happening Inside?

We've seen that the MLP works — it learns curved boundaries that logistic regression can't. But we've been treating it as a black box.

What actually happens when data flows through those layers? What do the hidden neurons compute? How does `loss.backward()` figure out which weights to blame?

**The next lesson opens the black box.** We'll:
- Trace real numbers through each layer, watching them transform
- Visualize what each neuron learns to detect
- Understand how gradients flow backward to update weights
- See the complete training loop in slow motion

You'll come away understanding not just *that* neural networks work, but *how* they work.

## Summary

**What we learned:**

1. **Classification** = predicting which group something belongs to (not a number)

2. **Logistic regression** = weighted sum → sigmoid → threshold. Simple, interpretable, but with basic features gives straight boundaries.

3. **Neural networks (MLPs)** = add hidden layers that automatically learn useful feature combinations. Can learn more complex patterns without manual feature engineering.

**The key insight:** Neural networks learn what features matter, instead of us having to figure it out manually.

**Next lesson:** We'll open the black box and see what's actually happening inside the neural network.

In [ ]:
print("="*55)
print("RESULTS")
print("="*55)
print(f"{'Model':<25} {'Validation Accuracy':<20}")
print("-"*45)
print(f"{'Logistic Regression':<25} {val_accuracy:<20.1%}")
print(f"{'MLP (16 hidden neurons)':<25} {mlp_val_acc:<20.1%}")
print()
print("Both work! The MLP is slightly better here.")
print("The real advantage of neural networks shows on")
print("more complex data (images, text) where manual")
print("feature engineering would be impractical.")

<div style="text-align: center; color: #888; font-size: 0.85em; margin-top: 40px; padding-top: 10px; border-top: 1px solid #ddd;">
© 2025 Utvecklarakademin UA Aktiebolag. All rights reserved.<br>
This material is proprietary and may not be reproduced, distributed, or shared without written permission.
</div>